# Day 2 agenda

- Recap + Q&A
- Magic methods
- Class attributes
- Finding attributes with the ICPO (instance-class-parents-object) rule
- Inheritance -- and how ICPO affects it
- Three models for understanding and using inheritance
- Data inheritance
- What next?

# Recap

Object-oriented programming is an organizational technique. It's all about packaging your code in a way that's easier to think about, easier to write, easier to maintain, and easier to read. 

1. A `class` is a new data type, one that exists in parallel with strings, lists, tuples, dicts, etc. Why create a new, specialized type, when you can just get by (and then some) with combinations of strings, lists, tuples, and dicts? As we saw yesterday, using a class instead of a tuple is easier to understand, and allows us to think at a higher level of abstraction. When you are programming with objects, the idea is to define new data types that reflect the nouns you'll want to manipulate, and methods (verbs) that you'll want to invoke on those new nouns.
2. Methods are functions that are attached to a particular class. We define one or more methods on our class, reflecting the actions that we want that class to take. Every class has different actions, and thus different methods.
3. A class name is not just the way we define the class, but also a "callable" that we can invoke to get a new value of that type. So if you want a new integer, you invoke `int`. If you want a new string, you invoke `str`. And if you want a new `Person`, you invoke `Person()`. We invoke the class, and get back a new instance of that class, a new object of that type.
4. When we create a new instance, Python takes care of allocating the memory and creating a new, "naked" object of that type. We get to influence the object after it was created, but before it's returned to the user in a special method known as `__init__`, for "initialization."
5. `__init__`, like all methods, gets `self` as the first parameter, which refers to the instance on which we're running. The job of `__init__` is to assign one or more attributes to `self`. These are known in other languages as "instance variables," because they're assigned to an instance.
6. The arguments that are passed to a class when we invoke it are handed to `__init__`, which can then use them to assign some of the new attributes when it's initializing an object.
7. `self` is always the first parameter, and thus always gets the instance on which we're running. You could (in theory) use any other name, but `self` is a very strong convention in the Python world. `self` is a local variable, because it's a parameter, so it only exists inside of a method. However, the object to which it refers continues to exist even after the method runs.


When we have the syntax

    a.b

that means: Go to the object `a` and retrieve its `b` attribute. But `a` and `b` can be anything! I could say:

    self.name = 'Reuven'   # set the attribute on self

Or I could say

    return self.name   # return the value associated with name, an attribute on self
    

What if I say this:

    self.books = []
    self.books.append('Python Workout')  # this appends to the list self.books, thanks to the method self.books.append
    self.books.append('Pandas Workout')

    

# Exercise: Cellphone

1. Define a `Cellphone` class. Each instance has two attributes:
    - `number`
    - `model`
2. You should be able to invoke the `call` method on an instance of `Cellphone`, passing the number you want to call. It'll return a string saying, `'Calling...'` and then print its own number and the number it's calling.

```
c = Cellphone('12345', 'G25+')
c.call('67890')  # 12345 is calling 67890
```

In [1]:
class Cellphone:
    def __init__(self, number, model):
        self.number = number
        self.model = model

c1 = Cellphone('12345', 'iPhone')
c2 = Cellphone('67890', 'Samsung')

In [2]:
c1.number

'12345'

In [3]:
c1.model

'iPhone'

In [4]:
vars(c1)

{'number': '12345', 'model': 'iPhone'}

In [5]:
vars(c2)

{'number': '67890', 'model': 'Samsung'}

In [6]:
class Cellphone:
    def __init__(self, number, model):
        self.number = number
        self.model = model
    def call(self, other_number):
        return f'Calling {other_number} from {self.number}, a {self.model} phone.'

c1 = Cellphone('12345', 'iPhone')
c2 = Cellphone('67890', 'Samsung')

In [8]:
c1.call(c2.number)

'Calling 67890 from 12345, a iPhone phone.'

In [9]:
c2.call(c1.number)   # this was rewritten, behind the scenes, to be: Cellphone.call(c2, c1.number)

'Calling 12345 from 67890, a Samsung phone.'

In [10]:
#             self   other_other
Cellphone.call(c2, c1.number)

'Calling 12345 from 67890, a Samsung phone.'

# Magic methods aka "dunder methods"

We've seen how we can implement methods on our objects:

- We write the method as part of the class body, with `def`
- We invoke the method (usually) via the instance

There are some places in Python where if a method by a certain name has been defined, then Python will invoke it for us, without asking. This is a protocol of sorts:

- We write the method
- We give the method a specific name that Python will look for
- Python invokes that method in particular circumstances

All of these special methods, also known as "magic methods," start and end with a double underscore, aka "dunder". These are thus known as "dunder methods."  There are many of these methods, but you don't have to define them all. 

Don't just define dunder methods on your own without checking what names you're using. Python won't stop you from defining any method with any name. But if you use a dunder name, and choose poorly, you can modify your object's behavior without realizing it.

In other words: Don't define dunder methods unless you know what name you're defining, and what Python will do when you define it.

In [11]:
class Person:
    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name
    def greet(self):
        return f'Hello, {self.first_name} {self.last_name}'

p = Person('Reuven', 'Lerner')
p.greet()

'Hello, Reuven Lerner'

In [12]:
# what happens if I now try to invoke the builtin len() function on my Person object?

len(p)

TypeError: object of type 'Person' has no len()

In [13]:
# how is it that we cannot get the len of a Person?


# To get the len, Python looks for `__len__`

Any class that wants to respond to the `len` builtin function needs to define the `__len__` dunder method. That method must return an integer. Plus, you aren't really supposed to invoke `__len__` or any other dunder method directly.

1. Define `__len__`
2. You're done.

In [14]:
class Person:
    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name
    def fullname(self):
        return f'{self.first_name} {self.last_name}'
    def greet(self):
        return f'Hello, {self.fullname()}'
    def __len__(self):
        return len(self.fullname())

p = Person('Reuven', 'Lerner')
p.greet()

'Hello, Reuven Lerner'

In [16]:
len(p) # len(p) -> p.__len__() -> p.fullname() -> len()

13

# Exercise: How many scoops?

Yesterday, we defined `Scoop` and `Bowl`. I want you to add `__len__` to `Bowl`, so that invoking `len` on a bowl returns the number of scoops.

In [27]:
class Scoop:
    def __init__(self, flavor):
        self.flavor = flavor

class Bowl:
    def __init__(self):
        self.scoops = []  
        self.price = 0

    def add_scoop(self, new_scoop):
        self.scoops.append(new_scoop)
        self.price = len(self.scoops) * 10

    def add_scoops(self, new_scoops):
        for one_scoop in new_scoops:
            self.add_scoop(one_scoop)

    def flavors(self):
        return [one_scoop.flavor
                for one_scoop in self.scoops]

    def __len__(self):
        return len(self.scoops)

    def total_price(self):
        return self.price

s1 = Scoop('chocolate')
s2 = Scoop('vanilla')
s3 = Scoop('persimmon')

b1 = Bowl()
b1.add_scoops([s1, s2])

b2 = Bowl()
b2.add_scoops([s3, s1, s1])

print(b1.flavors())

['chocolate', 'vanilla']


In [28]:
print(b2.flavors())

['persimmon', 'chocolate', 'chocolate']


In [29]:
len(b1)

2

In [30]:
len(b2)

3

Repeating dunder methods...

Often, a dunder method is very specific to a class and its attributes, and so you can't easily share them across classes.

But if you *can*, then that's where *inheritance* comes in. 

In [31]:
b1.total_price()

20

In [32]:
b2.total_price()

30

# You can go wild!

If you want, you can define `__len__` to be anything useful, or silly, or interesting. So long as it returns an integer, it can execute any code. You'll definitely find some people who abuse the dunder methods a bit too much.

For example, you could use `len` to retrieve something that has nothing to do with the length, but is an integer. For example, you could define `Person.__len__` to return the person's height, or their bank account balance.

But if you invoke `len(p1)` and you get their bank balance, that seems ... weird.

# Next up

1. `__str__` and `__repr__`
2. Other magic methods
3. Class attributes -- what are they, and how do they work?

In [33]:
# let's look at s1 -- what happens when I ask for s1.flavor?

s1.flavor

'chocolate'

In [36]:
print(s1)  # let's print out our instance of Scoop

Why do we get something so ugly? What is this?

- `__main__` is the namespace we're in -- the first file that Python loaded (i.e., Jupyter notebook)
- `Scoop` is the class
- The long hex number is the ID of the object -- also the location of the object in our computer's memory

In [37]:
# every object in Python knows how to display itself in `print`

print(5)

5


In [38]:
print([10, 20, 30])

[10, 20, 30]


In [39]:
print({'a':10, 'b':20})

{'a': 10, 'b': 20}


- `print` only knows how to print strings
- When you pass a value to `print`, it invokes `str` on that value, getting a new string, which it can print
- So `str` knows how to turn each value into a string? NO!
- `str` turns to the object, and invokes its `__str__` method. Whatever the object returns is printed for that object

If we implement `__str__` on our object, then we can dictate what we'll get back when turning our object into a string.

In [42]:
class Person:
    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name
    def fullname(self):
        return f'{self.first_name} {self.last_name}'
    def greet(self):
        return f'Hello, {self.fullname()}'
    def __len__(self):
        return len(self.fullname())
    def __str__(self):   # __str__ takes no arguments
        return f'Person named {self.fullname()}'

p = Person('Reuven', 'Lerner')
print(p)   # we'll get the same ugly default as we did for Scoop

Person named Reuven Lerner


If you want your object to be displayable by `print` in a nice way, you can define `__str__` in your class.

# Exercise: Printable scoops

Modify the `Scoop` class, such that turning an instance of `Scoop` into a string with `str` returns something like "Scoop of FLAVOR".

In [48]:
class Scoop:
    def __init__(self, flavor):
        self.flavor = flavor

    def __str__(self):
        return f'Scoop of {self.flavor}!!!!'

class Bowl:
    def __init__(self):
        self.scoops = []  
        self.price = 0

    def add_scoop(self, new_scoop):
        self.scoops.append(new_scoop)
        self.price = len(self.scoops) * 10

    def add_scoops(self, new_scoops):
        for one_scoop in new_scoops:
            self.add_scoop(one_scoop)

    def flavors(self):
        return [one_scoop.flavor
                for one_scoop in self.scoops]

    def __len__(self):
        return len(self.scoops)

    def total_price(self):
        return self.price

In [50]:
s1 = Scoop('chocolate')
s2 = Scoop('vanilla')
s3 = Scoop('persimmon')

In [51]:
print(s1) # print(str(s1)) -> print(s1.__str__())
print(s2)
print(s3)

Scoop of chocolate!!!!
Scoop of vanilla!!!!
Scoop of persimmon!!!!


# Exercise: Printable bowls

1. Add `__str__` to the `Bowl` class
2. Printing an instance of `Bowl` should show:
    - It identifies as a `Bowl`
    - It prints each of the scoops in that bowl
    - It numbers the scoops, starting with 1

If I say

    print(b)

I should see

    Bowl with:
    1. Scoop of chocolate
    2. Scoop of vanilla
    3. Scoop of persimmon
    

In [62]:
d

['chocolate', 'vanilla', 'persimmon']


In [63]:
print(b)  # I want to see the bowl's contents here

Bowl of: 
	1. Scoop of chocolate
	2. Scoop of vanilla
	3. Scoop of persimmon



In [64]:
# there is some bad news...

2 + 3

5

In [65]:
[10, 20, 30] + [40, 50, 60]

[10, 20, 30, 40, 50, 60]

In [66]:
s1  # why are we getting this? Shouldn't we be getting our nice, new __str__ output? No.

# Two ways to print objects

Every object in Python has *two* ways to be turned into a string:

- The way that we normally think about, and which is meant for normal people, is `str`, handled by the `__str__` method.
- There is a second way, meant for programmers in Jupyter, the Python debugger, and other internal areas. This version is known as `repr`, and is handled by the `__repr__` method.

In our `Scoop` and `Bowl` cases, we defined `__str__` on both, but we didn't define `__repr__` at all. For this reason, the default `__repr__` was invoked, and it's almost identical to the default `__str__`.

If we only define `__repr__`, then it handles *both* `__repr__` and `__str__`. So... why not just define `__repr__`?

First: I do that, and I encourage you to do that. If and when you ever want to distinguish between the output for programmers and the output for normal users, you can always define `__str__`.

There is a convention in Python that `__repr__` should return legal Python code. I reject this. 



In [67]:
class Scoop:
    def __init__(self, flavor):
        self.flavor = flavor

    def __repr__(self):
        return f'Scoop of {self.flavor}'

class Bowl:
    def __init__(self):
        self.scoops = []  
        self.price = 0

    def add_scoop(self, new_scoop):
        self.scoops.append(new_scoop)
        self.price = len(self.scoops) * 10

    def add_scoops(self, new_scoops):
        for one_scoop in new_scoops:
            self.add_scoop(one_scoop)

    def flavors(self):
        return [one_scoop.flavor
                for one_scoop in self.scoops]

    def __len__(self):
        return len(self.scoops)

    def total_price(self):
        return self.price

    def __repr__(self):
        output = f'Bowl of: \n'

        for index, one_scoop in enumerate(self.scoops):
            output += f'\t{index+1}. {one_scoop}\n'

        return output
        

s1 = Scoop('chocolate')
s2 = Scoop('vanilla')
s3 = Scoop('persimmon')

b = Bowl()
b.add_scoops([s1, s2])
b.add_scoop(s3)

print(b.flavors())

['chocolate', 'vanilla', 'persimmon']


In [68]:
print(b)

Bowl of: 
	1. Scoop of chocolate
	2. Scoop of vanilla
	3. Scoop of persimmon



In [69]:
b

Bowl of: 
	1. Scoop of chocolate
	2. Scoop of vanilla
	3. Scoop of persimmon

In [70]:
s1

Scoop of chocolate

# Another magic method: `__getitem__`

You might have noticed that:

- You retrieve from a string with `[]`
- You retrieve from a list with `[]`
- You retrieve from a tuple with `[]`
- You retrieve from a dict with `[]`

The reason is that these classes all defined the `__getitem__` method. This method takes two arguments, self and the index. 

Each class can define `__getitem__` however they want. Whatever was inside of the `[]` is passed as the `index` argument.

In [71]:
class Person:
    def __init__(self, name):
        self.name = name

    def __getitem__(self, index):
        return self.name[index]

In [73]:
p = Person('Reuven')

p[0]  # translated into p.__getitem__(0) -> Person.__getitem__(p, 0)

'R'

In [75]:
p[1:4] #this works, because __getitem__ passes it along to `[]`

'euv'

# Exercise: Subscriptable bowls

1. Add `__getitem__` to the bowl class
2. It should be possible to get the scoop at the index you want
3. It's OK to raise an exception (or let Python do so) if you're out of bounds

In [82]:
class Scoop:
    def __init__(self, flavor):
        self.flavor = flavor

    def __repr__(self):
        return f'Scoop of {self.flavor}'

class Bowl:
    def __init__(self):
        self.scoops = []  
        self.price = 0

    def add_scoop(self, new_scoop):
        self.scoops.append(new_scoop)
        self.price = len(self.scoops) * 10

    def add_scoops(self, new_scoops):
        for one_scoop in new_scoops:
            self.add_scoop(one_scoop)

    def flavors(self):
        return [one_scoop.flavor
                for one_scoop in self.scoops]

    def __len__(self):
        return len(self.scoops)

    def total_price(self):
        return self.price

    def __repr__(self):
        output = f'Bowl of: \n'

        for index, one_scoop in enumerate(self.scoops):
            output += f'\t{index+1}. {one_scoop}\n'

        return output

    def __getitem__(self, index):
        return self.scoops[index]
        

s1 = Scoop('chocolate')
s2 = Scoop('vanilla')
s3 = Scoop('persimmon')

b = Bowl()
b.add_scoops([s1, s2])
b.add_scoop(s3)

print(b.flavors())

print(b[0])  # -> print(b.__getitem__(0))
print(b[2])
print(b[1:]) # -> print(b.__getitem__(slice(1, None)))

['chocolate', 'vanilla', 'persimmon']
Scoop of chocolate
Scoop of persimmon
[Scoop of vanilla, Scoop of persimmon]


In [81]:
b

The dunder methods are special because Python knows to look for them. If you define `__teacup__`, then Python won't object, but it also won't use the method in any special way, because it's not expecting it.

If you mean, "Can you define dunder methods?" Yes. If you mean, "Can you define them if Python doesn't know what to do with them," the answer is still Yes. But Python will just ignore the ones it doesn't know.

Every dunder method in Python: https://www.pythonmorsels.com/every-dunder-method/

In [79]:
# if you iterate over enumerate(SOMETHING), then
# each iteration gives you TWO values, the index (starting at 0)
# and the value you would normally have gotten

for index, one_character in enumerate('abcd'):
    print(f'{index}: {one_character}')

0: a
1: b
2: c
3: d


# Another dunder method: `__eq__`

This method also takes two arguments, `self` and `other`. This is invoked whenever you see the `==` operator. Whatever is on the *left* side of `==` has the method invoked.

In [83]:
class Person:
    def __init__(self, name):
        self.name = name

p1 = Person('someone')
p2 = Person('someone')

p1 == p2

False

In [84]:
class Person:
    def __init__(self, name):
        self.name = name
    def __eq__(self, other):
        return self.name == other.name

p1 = Person('someone')
p2 = Person('someone')

p1 == p2

True

In [85]:
p1 == 5

AttributeError: 'int' object has no attribute 'name'

In [86]:
5 == p1

AttributeError: 'int' object has no attribute 'name'

# Exercise: Ice cream equality

1. Add `__eq__` to `Scoop`, and check that two scoops with the same flavor (string) are equal.
2. Add `__eq__` to `Bowl`. Two bowls are the same if they have the same number of scoops, and the flavors are the same, too.

In [95]:
class Scoop:
    def __init__(self, flavor):
        self.flavor = flavor

    def __repr__(self):
        return f'Scoop of {self.flavor}'

    def __eq__(self, other):
        return self.flavor == other.flavor

class Bowl:
    def __init__(self):
        self.scoops = []  
        self.price = 0

    def add_scoop(self, new_scoop):
        self.scoops.append(new_scoop)
        self.price = len(self.scoops) * 10

    def add_scoops(self, new_scoops):
        for one_scoop in new_scoops:
            self.add_scoop(one_scoop)

    def flavors(self):
        return [one_scoop.flavor
                for one_scoop in self.scoops]

    def __len__(self):
        return len(self.scoops)

    def total_price(self):
        return self.price

    def __repr__(self):
        output = f'Bowl of: \n'

        for index, one_scoop in enumerate(self.scoops):
            output += f'\t{index+1}. {one_scoop}\n'

        return output

    def __getitem__(self, index):
        return self.scoops[index]

    def __eq__(self, other):
        return sorted(self.flavors()) == sorted(other.flavors())
        

s1 = Scoop('chocolate')
s2 = Scoop('chocolate')

b1 = Bowl()
b1.add_scoops([s1, s2])
b1.add_scoop(s3)

b2 = Bowl()
b2.add_scoops([s2, s3])
b2.add_scoop(s1)

print(s1 == s2)

True


In [96]:
print(b1 == b2)

True


In [100]:
b1 = Bowl()
b1.add_scoops([s1, s3])
b1.add_scoops([s1, s3])

b2 = Bowl()
b2.add_scoops([s1, s3])

b1 == b2

False

In [101]:
b1.flavors()

['chocolate', 'persimmon', 'chocolate', 'persimmon']

In [102]:
b2.flavors()

['chocolate', 'persimmon']

# Next up

1. Class attributes
2. Attribute lookup
3. Inheritance

# Everything is an object!

What does it mean that everything is an object?

- Everything has a class
- Everything has *attributes*. These are the private storage area that every value in Python has, which we read from and write to with `.`

What has attributes? 

- Strings, lists, and tuples -- they all have attributes
- scoops and bowls -- they do, too!
- functions -- yes
- classes -- yes

In [103]:
# our company has a Person class that our customers use

class Person:
    def __init__(self, name):
        self.name = name

    def greet(self):
        return f'Hello, {self.name}!'

p1 = Person('name1')
p2 = Person('name2')

print(p1.greet())
print(p2.greet())

Hello, name1!
Hello, name2!


Some of our customers would like new functionality -- they want to know how many people have been created in their virtual universe.
If we've created five Person objects, it should be known.

We could use a global variable to keep track right?

In [105]:
# population-tracking version 1

population = 0

class Person:
    def __init__(self, name):
        global population  # make sure that population is seen as a global variable, not a local one
        self.name = name
        population += 1    # add 1 to the global population

    def greet(self):
        return f'Hello, {self.name}!'

print(f'Before, population = {population}')
p1 = Person('name1')
p2 = Person('name2')
print(f'After, population = {population}')

print(p1.greet())
print(p2.greet())

Before, population = 0
After, population = 2
Hello, name1!
Hello, name2!


In [106]:
# population-tracking version 2

class Person:
    def __init__(self, name):
        self.name = name
        Person.population += 1    

    def greet(self):
        return f'Hello, {self.name}!'

Person.population = 0           # add a new attribute to the Person class

print(f'Before, population = {Person.population}')
p1 = Person('name1')
p2 = Person('name2')
print(f'After, population = {Person.population}')

print(p1.greet())
print(p2.greet())

Before, population = 0
After, population = 2
Hello, name1!
Hello, name2!


In [107]:
# population-tracking version 3 -- the best, most Pythonic way

# when we use "def", we normally assign the new function to variable
# but inside of a class definition, we actually create a new attribute on Person, not a new variable

class Person:
    population = 0  # this is really Person.population = 0 -- same as line 11 in the above cell, but while Person is being defined, not after

    def __init__(self, name):     # this is really Person.__init__
        self.name = name
        Person.population += 1    

    def greet(self):               # this is really Person.greet
        return f'Hello, {self.name}!'

print(f'Before, population = {Person.population}')
p1 = Person('name1')
p2 = Person('name2')
print(f'After, population = {Person.population}')

print(p1.greet())
print(p2.greet())

Before, population = 0
After, population = 2
Hello, name1!
Hello, name2!


# Class attributes

Every object in Python has attributes, and that includes classes. We can add a new attribute to our class by assigning whenever we want. But normally, we'll want to put the assignment inside of the class definition. 

Why would we want this?

There might be values that we want to store in a commonly known and accepted place.

We also might have constant values that everyone will want to use.

In both cases, storing an attribute on the class is better than a global variable.

In [108]:
class Person:
    population = 0  # maybe this is shared among the class and its instances

    def __init__(self, name):     # this is really Person.__init__
        self.name = name
        Person.population += 1    

    def greet(self):               # this is really Person.greet
        return f'Hello, {self.name}!'

print(f'Before, population = {Person.population}')
p1 = Person('name1')
p2 = Person('name2')
print(f'After, population = {Person.population}')
print(f'After, p1.population = {p1.population}')
print(f'After, p2.population = {p2.population}')

print(p1.greet())
print(p2.greet())

Before, population = 0
After, population = 2
After, p1.population = 2
After, p2.population = 2
Hello, name1!
Hello, name2!


In [109]:
Person.population   # I'm outside of the class, and yet...

2

How can it be that even though population was defined on Person, we can access it via p1 and p2?

The answer: Python's attribute lookup, which I call ICPO.

- We ask for `p1.population`
    - Python asks `p1`: Do you have an attribute named `population`? Answer: no.
    - Python asks `p1`'s class: Do you have an attribute named `population`? Answer: yes! It returns 2

- First, Python checks with the instance (I)
- Then, if the attribute isn't there, it checks on the class, C.

What does this explain?

Method invocations. If I invoke `p1.greet`, then what happens?

- Python asks `p1`: Do you have an attribute named `greet`? Answer: No.
- Python asks `p1`'s class, `Person`: Do you have an attribute named `greet`? Yes!

We get the method from the class, and invoke it.

# Exercise: Limiting scoops

We're changing the rules at our ice cream shop! Until now, you could have any number of scoops in a bowl. As of now, the maximum is 5 scoops. That number should be set in `MAX_SCOOPS`, a class attribute in `Bowl`.

1. Add the class attribute.
2. Modify `add_scoop` (singular) such that it'll only really add if the current number of scoops is below that threshold.

In [114]:
class Scoop:
    def __init__(self, flavor):
        self.flavor = flavor

    def __repr__(self):
        return f'Scoop of {self.flavor}'

    def __eq__(self, other):
        return self.flavor == other.flavor

class Bowl:
    MAX_SCOOPS = 5  # this defines Bowl.MAX_SCOOPS
    
    def __init__(self):
        self.scoops = []  
        self.price = 0

    def add_scoop(self, new_scoop):
        if len(self.scoops) < self.MAX_SCOOPS:
            self.scoops.append(new_scoop)
            self.price = len(self.scoops) * 10

    def add_scoops(self, new_scoops):
        for one_scoop in new_scoops:
            self.add_scoop(one_scoop)

    def flavors(self):
        return [one_scoop.flavor
                for one_scoop in self.scoops]

    def __len__(self):
        return len(self.scoops)

    def total_price(self):
        return self.price

    def __repr__(self):
        output = f'Bowl of: \n'

        for index, one_scoop in enumerate(self.scoops):
            output += f'\t{index+1}. {one_scoop}\n'

        return output

    def __getitem__(self, index):
        return self.scoops[index]

    def __eq__(self, other):
        return sorted(self.flavors()) == sorted(other.flavors())
        

s1 = Scoop('chocolate')
s2 = Scoop('vanilla')
s3 = Scoop('persimmon')
s4 = Scoop('four')
s5 = Scoop('five')
s6 = Scoop('six')

b = Bowl()
b.add_scoops([s1, s2])
b.add_scoops([s3, s4, s5])
b.add_scoop(s6)

print(b)

Bowl of: 
	1. Scoop of chocolate
	2. Scoop of vanilla
	3. Scoop of persimmon
	4. Scoop of four
	5. Scoop of five



# Inheritance!

Inheritance is a major concept in object-oriented programming. It means:

- You define a new class
- You indicate that your new class *inherits* from another one
- When an attribute isn't found on your class, Python looks on the parent

When would you do this?

- A class already exists that does some (most?) of what you want
- Instead of reinventing the wheel, you say, "I have a class that resembles that one."
- A class that inherits from another is a "child class."
- A class that has children is a "parent class."

This can also be useful if you want to modify the functionality in a class, but don't have permission to do so. You just "subclass it," and then use it.

In [115]:
class Person:
    def __init__(self, name):   
        self.name = name

    def greet(self):            
        return f'Hello, {self.name}!'

p1 = Person('name1')
p2 = Person('name2')

print(p1.greet())
print(p2.greet())

Hello, name1!
Hello, name2!


What if our customers now want an `Employee` class that does the same thing as `Person`, but also has a second attribute, `employee_id`.

One way to do this is copy and paste, modifying `Employee`.

Instead, we can define `Employee` as a class that *inherits from `Person`*.

In [118]:
class Employee(Person):   # this means: Employee is-a Person
    def __init__(self, name, employee_id):
        self.name = name
        self.empoyee_id = employee_id

e1 = Employee('emp1', 1)  #Python asked e1 if it has 'greet'. No. It asked e1's class, Employee. It also lacked one.
                          # Python asked e1's class's parent (i.e., Person). It had one
e2 = Employee('emp2', 2)

In [119]:
e1.greet()

'Hello, emp1!'

# Next up

- Finalizing our understanding of inheritance
- Using `super` when inheriting
- `object`, and its role
- Three pardigms of method inheritance

In [120]:
_secret = 'shhh!'  # this is as close as we come to private/protected

# Relationships

- has-a relationship is composition, where one object "owns" another. This is wherever you have an attribute.
- is-a relationship is inheritance, where a subclass "is" also its parent.

In [123]:
class Person:
    def __init__(self, name):   
        self.name = name

    def greet(self):            
        return f'Hello, {self.name}!'

p1 = Person('name1')
p2 = Person('name2')

print(p1.greet())
print(p2.greet())


class Employee(Person):  
    def __init__(self, name, employee_id):
        # self.name = name
        self.empoyee_id = employee_id

e1 = Employee('emp1', 1) # Python looks for __init__ -- first on the instance (not there), then on the class (Employee)
e2 = Employee('emp2', 2)

print(e1.greet())  # Employee doesn't have greet -- but Person does
print(e2.greet())  # same here

Hello, name1!
Hello, name2!


AttributeError: 'Employee' object has no attribute 'name'

`__init__` is not a set of declarations of what attributes must be defined on a given object. Rather, it's a method that must run in order to set those attributes.

If `Employee.__init__` exists, then that is what will be invoked when we create a new instance of `Employee`. We will never get to `Person.__init__` unless we explicitly ask to do so.

In [124]:
class Person:
    def __init__(self, name):   
        self.name = name

    def greet(self):            
        return f'Hello, {self.name}!'

p1 = Person('name1')
p2 = Person('name2')

print(p1.greet())
print(p2.greet())


class Employee(Person):  
    def __init__(self, name, employee_id):
        Person.__init__(self, name)  # let's explicitly invoke Person.__init__!
        self.empoyee_id = employee_id

e1 = Employee('emp1', 1) 
e2 = Employee('emp2', 2)

print(e1.greet())  
print(e2.greet())  

Hello, name1!
Hello, name2!
Hello, emp1!
Hello, emp2!


If the child class's method is doing some of the same things as the parent class's method, then why aren't you invoking the parent's method? Re-implementing the same method is exactly what we're trying to avoid with objects!

One easy (but long) way to do this is to invoke `Person.__init__` at the start of `Employee.__init__`. It'll do its thing, and then return, and then `Employee.__init__` has the opportunity to add its attributes.

But this seems a bit clunky. There is a better way, though: `super`.

In [125]:
class Person:
    def __init__(self, name):   
        self.name = name

    def greet(self):            
        return f'Hello, {self.name}!'

p1 = Person('name1')
p2 = Person('name2')

print(p1.greet())
print(p2.greet())


class Employee(Person):  
    def __init__(self, name, employee_id):
        super().__init__(name)  # we invoke super(), we pass the name of the method to run, and the non-self argument(s)
        self.empoyee_id = employee_id

e1 = Employee('emp1', 1) 
e2 = Employee('emp2', 2)

print(e1.greet())  
print(e2.greet())  

Hello, name1!
Hello, name2!
Hello, emp1!
Hello, emp2!


# Three paradigms of method inheritance

1. The child class does nothing, and the parent class provides the method. We see that here with `greet`. Here, the child lets the parent's implementation still work.
2. The child class implements something totally new and different. Here, we basically ignore the parent class.
3. The child method first invokes the parent (with `super()`), then does its own thing.

# Exercise: Big Bowl

Implement a `BigBowl` class that is precisely the same as `Bowl`, but can handle 7 scoops of ice cream.

- Change as little as possible (if anything) from `Bowl`
- Write as little as possible in `BigBowl`

In [129]:
class Scoop:
    def __init__(self, flavor):
        self.flavor = flavor

    def __repr__(self):
        return f'Scoop of {self.flavor}'

    def __eq__(self, other):
        return self.flavor == other.flavor

class Bowl:
    MAX_SCOOPS = 5  # this defines Bowl.MAX_SCOOPS
    
    def __init__(self):
        self.scoops = []  
        self.price = 0

    def add_scoop(self, new_scoop):
        if len(self.scoops) < self.MAX_SCOOPS:
            self.scoops.append(new_scoop)
            self.price = len(self.scoops) * 10

    def add_scoops(self, new_scoops):
        for one_scoop in new_scoops:
            self.add_scoop(one_scoop)

    def flavors(self):
        return [one_scoop.flavor
                for one_scoop in self.scoops]

    def __len__(self):
        return len(self.scoops)

    def total_price(self):
        return self.price

    def __repr__(self):
        output = f'Bowl of: \n'

        for index, one_scoop in enumerate(self.scoops):
            output += f'\t{index+1}. {one_scoop}\n'

        return output

    def __getitem__(self, index):
        return self.scoops[index]

    def __eq__(self, other):
        return sorted(self.flavors()) == sorted(other.flavors())
        
class BigBowl(Bowl):
    MAX_SCOOPS = 7  # this defines BigBowl.MAX_SCOOPS

s1 = Scoop('chocolate')
s2 = Scoop('vanilla')
s3 = Scoop('persimmon')
s4 = Scoop('four')
s5 = Scoop('five')
s6 = Scoop('six')
s7 = Scoop('seven')
s8 = Scoop('eight')

bb = BigBowl()
bb.add_scoops([s1, s2])
bb.add_scoops([s3, s4, s5])
bb.add_scoops([s6, s7, s8])

print(bb)

Bowl of: 
	1. Scoop of chocolate
	2. Scoop of vanilla
	3. Scoop of persimmon
	4. Scoop of four
	5. Scoop of five
	6. Scoop of six
	7. Scoop of seven



# The full attribute search path

If you ask for an attribute on an object in Python, here's where Python really looks:

- `I` -- the instance, the value you specifically name
- `C` -- the class, if it's not on the instance
- `P` -- the parent, if it's not on the class
- `O` -- `object`, the top of our class hierarchy

What is defined on `object`?
- `__str__`
- `__repr__`

When we originally said `print(s1)` to an instance of `Scoop`, we were 